In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_5.py ──
"""
Shared infrastructure for MLFP04 Exercise 5 — Association Rules.

Contains: synthetic Singapore retail transaction generator, category map,
rule dataclass helpers, and small polars utilities used by every technique
file in ``modules/mlfp04/solutions/ex_5/``.

Technique-specific code (Apriori from scratch, FP-Growth wrapper, rule
evaluation, feature engineering for classification) does NOT belong here —
it lives in the per-technique files.
"""

from collections import defaultdict
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp04_ex5_association_rules"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE RETAIL PRODUCT CATALOGUE
# ════════════════════════════════════════════════════════════════════════
# 25 products grouped to mirror a typical HDB neighbourhood mini-mart basket.

PRODUCTS: list[str] = [
    "bread",
    "butter",
    "milk",
    "eggs",
    "rice",
    "noodles",
    "soy_sauce",
    "cooking_oil",
    "chicken",
    "fish",
    "coffee",
    "tea",
    "sugar",
    "condensed_milk",
    "biscuits",
    "chips",
    "soft_drink",
    "beer",
    "wine",
    "tissue",
    "shampoo",
    "soap",
    "detergent",
    "toothpaste",
    "bananas",
]

CATEGORY_MAP: dict[str, str] = {
    "bread": "breakfast",
    "butter": "breakfast",
    "eggs": "breakfast",
    "milk": "dairy",
    "condensed_milk": "dairy",
    "coffee": "beverage",
    "tea": "beverage",
    "soft_drink": "beverage",
    "sugar": "pantry",
    "rice": "pantry",
    "cooking_oil": "pantry",
    "soy_sauce": "pantry",
    "noodles": "pantry",
    "chicken": "protein",
    "fish": "protein",
    "beer": "alcohol",
    "wine": "alcohol",
    "chips": "snack",
    "biscuits": "snack",
    "bananas": "fruit",
    "shampoo": "personal_care",
    "soap": "personal_care",
    "toothpaste": "personal_care",
    "tissue": "household",
    "detergent": "household",
}

# Co-purchase bundles (items, probability). Models real behaviour:
# kaya-toast breakfast, kopi-C, household replenishment, beer+chips, etc.
BUNDLES: list[tuple[list[str], float]] = [
    (["bread", "butter", "eggs"], 0.15),
    (["coffee", "condensed_milk", "sugar"], 0.12),
    (["rice", "chicken", "soy_sauce"], 0.10),
    (["noodles", "eggs", "soy_sauce"], 0.08),
    (["beer", "chips"], 0.09),
    (["milk", "biscuits"], 0.07),
    (["shampoo", "soap", "toothpaste"], 0.06),
    (["tea", "sugar", "biscuits"], 0.05),
    (["wine", "chips", "biscuits"], 0.04),
    (["cooking_oil", "rice", "fish"], 0.06),
    (["detergent", "tissue", "soap"], 0.05),
    (["bananas", "milk", "eggs"], 0.05),
]

N_TRANSACTIONS_DEFAULT: int = 2500


# ════════════════════════════════════════════════════════════════════════
# TRANSACTION GENERATOR
# ════════════════════════════════════════════════════════════════════════


def generate_transactions(
    n: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> list[set[str]]:
    """Generate synthetic Singapore retail transactions.

    Each transaction is a set of product strings. Bundles fire with their
    listed probability; each item inside a firing bundle is kept with 0.85
    probability (random drop-out) so support is noisy. A Poisson number of
    random items is added on top to simulate impulse buys.
    """
    rng = np.random.default_rng(seed)
    transactions: list[set[str]] = []
    for _ in range(n):
        basket: set[str] = set()
        for bundle_items, prob in BUNDLES:
            if rng.random() < prob:
                for item in bundle_items:
                    if rng.random() < 0.85:
                        basket.add(item)
        n_random = rng.poisson(2)
        if n_random > 0:
            random_items = rng.choice(
                PRODUCTS, size=int(min(n_random, 5)), replace=False
            )
            basket.update(random_items)
        if basket:
            transactions.append(basket)
    return transactions


def transactions_to_onehot(transactions: list[set[str]]) -> pl.DataFrame:
    """One-row-per-transaction boolean matrix (columns = sorted PRODUCTS).

    Polars-native. Used as input to mlxtend FP-Growth (via .to_pandas()).
    """
    all_items = sorted(PRODUCTS)
    rows = [{item: (item in txn) for item in all_items} for txn in transactions]
    return pl.DataFrame(rows)


def product_frequency(transactions: Iterable[set[str]]) -> dict[str, int]:
    """Count how many transactions contain each product."""
    counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            counts[item] += 1
    return dict(counts)


def print_transaction_summary(transactions: list[set[str]]) -> None:
    """One-liner summary + top 10 product frequency. Used by every file."""
    avg_basket = float(np.mean([len(t) for t in transactions]))
    print("=== Synthetic Singapore Retail Transactions ===")
    print(f"  Transactions: {len(transactions):,}")
    print(f"  Products:     {len(PRODUCTS)}")
    print(f"  Avg basket:   {avg_basket:.1f} items")

    freq = product_frequency(transactions)
    n = len(transactions)
    print("\n  Top 10 products by frequency:")
    for item, count in sorted(freq.items(), key=lambda kv: -kv[1])[:10]:
        print(f"    {item:<20} {count:>5} ({count / n:.1%})")


# ════════════════════════════════════════════════════════════════════════
# RULE HELPERS
# ════════════════════════════════════════════════════════════════════════


def format_itemset(items: Iterable[str]) -> str:
    """Deterministic pretty-print of a frozenset of items."""
    return ", ".join(sorted(items))


def categorise_rule(
    antecedent: frozenset[str],
    consequent: frozenset[str],
) -> tuple[set[str], set[str], str]:
    """Return (antecedent_categories, consequent_categories, relation_type)."""
    ant_cats = {CATEGORY_MAP.get(item, "other") for item in antecedent}
    con_cats = {CATEGORY_MAP.get(item, "other") for item in consequent}
    if ant_cats == con_cats:
        rel = "within-category complement"
    elif ant_cats & con_cats:
        rel = "cross-category with overlap"
    else:
        rel = "cross-category association"
    return ant_cats, con_cats, rel


def rules_to_polars(rules: list[dict]) -> pl.DataFrame:
    """Convert a list of rule dicts into a polars DataFrame for plotting."""
    return pl.DataFrame(
        {
            "antecedent": [format_itemset(r["antecedent"]) for r in rules],
            "consequent": [format_itemset(r["consequent"]) for r in rules],
            "support": [float(r["support"]) for r in rules],
            "confidence": [float(r["confidence"]) for r in rules],
            "lift": [float(r["lift"]) for r in rules],
        }
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 5.1: Apriori Algorithm from Scratch
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement the Apriori algorithm for frequent itemset mining
#   - Apply the anti-monotone pruning principle by hand
#   - Generate candidate k-itemsets from frequent (k-1)-itemsets
#   - Count support with a single pass per level
#   - Understand why Apriori scales beyond brute-force enumeration
#
# PREREQUISITES:
#   - MLFP04 Exercise 1 (clustering — pattern discovery without labels)
#   - Basic set theory (subset, superset, intersection)
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — why Apriori works (anti-monotone pruning)
#   2. Build — implement `apriori()` and `_generate_candidates()`
#   3. Train — run it on 2,500 Singapore retail transactions
#   4. Visualise — L1 -> L2 -> L3 ladder and top frequent itemsets
#   5. Apply — FairPrice/Sheng Siong shelf layout optimisation
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from collections import defaultdict

import polars as pl



## THEORY — Why Apriori Works


In [ ]:
# With 25 products there are 2^25 ~= 33 million possible itemsets. A naive
# algorithm would test every one of them against every transaction. Apriori
# avoids that with a single observation:
#
#   ANTI-MONOTONE PRINCIPLE
#   If an itemset X is INFREQUENT, then every superset of X is ALSO
#   infrequent. Reason: if fewer than min_count baskets contain X, then
#   even fewer baskets contain X plus any extra item.
#
# Consequence: once L_k (frequent k-itemsets) is known, we only need to
# form candidates for L_{k+1} by joining L_k with itself AND requiring
# every (k)-subset of the new candidate to already be in L_k. Any itemset
# with an infrequent subset is pruned before it is ever counted.
#
# COST:
#   Level 1: scan once, count single items.  O(N * |I|)
#   Level k: generate C_k from L_{k-1}, scan to count, filter.
#   Terminates when L_k is empty.
#
# On a 2,500-transaction basket with 25 products, Apriori evaluates on
# the order of ~1,000 candidates — a 30,000x reduction from brute force.



## TASK 2 — BUILD: Apriori from scratch


Applies the anti-monotone pruning rule: every (k-1)-subset of a
    candidate MUST already be in `prev_level`, otherwise drop the
    candidate without ever counting it.


Returns ``{itemset: support}`` for every itemset whose support meets
    ``min_support``. Support is the fraction of baskets containing the
    itemset.


In [ ]:
def _generate_candidates(
    prev_level: list[frozenset[str]],
    k: int,
) -> list[frozenset[str]]:
    prev_set = set(prev_level)
    candidates: set[frozenset[str]] = set()

    for i, a in enumerate(prev_level):
        for b in prev_level[i + 1 :]:
            union = a | b
            if len(union) != k:
                continue
            all_subsets_frequent = all(
                (union - frozenset([item])) in prev_set for item in union
            )
            if all_subsets_frequent:
                candidates.add(union)

    return list(candidates)


def apriori(
    transactions: list[set[str]],
    min_support: float,
    verbose: bool = True,
) -> dict[frozenset[str], float]:
    n = len(transactions)
    min_count = min_support * n

    # ── L1: frequent single items ────────────────────────────────────
    item_counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            item_counts[item] += 1

    freq_itemsets: dict[frozenset[str], float] = {}
    current_level: list[frozenset[str]] = []
    for item, count in item_counts.items():
        if count >= min_count:
            fs = frozenset([item])
            freq_itemsets[fs] = count / n
            current_level.append(fs)

    if verbose:
        print(f"  L1: {len(current_level)} frequent items (min_support={min_support})")

    # ── L2, L3, ... Lk: joined + pruned candidates ───────────────────
    k = 2
    while current_level:
        candidates = _generate_candidates(current_level, k)
        if not candidates:
            break

        candidate_counts: dict[frozenset[str], int] = defaultdict(int)
        for txn in transactions:
            txn_frozen = frozenset(txn)
            for candidate in candidates:
                if candidate.issubset(txn_frozen):
                    candidate_counts[candidate] += 1

        current_level = []
        for candidate, count in candidate_counts.items():
            if count >= min_count:
                freq_itemsets[candidate] = count / n
                current_level.append(candidate)

        if verbose:
            print(f"  L{k}: {len(current_level)} frequent {k}-itemsets")
        k += 1

    return freq_itemsets



## TASK 3 — TRAIN: run Apriori on Singapore retail baskets


In [ ]:
transactions = generate_transactions(n=2500, seed=42)
print_transaction_summary(transactions)

print("\n=== Apriori (from scratch) ===")
MIN_SUPPORT = 0.03
frequent_itemsets = apriori(transactions, min_support=MIN_SUPPORT)
print(f"\n  Total frequent itemsets: {len(frequent_itemsets)}")



### Checkpoint


In [ ]:
assert len(frequent_itemsets) > 0, "Apriori should find at least one frequent itemset"
n_txn = len(transactions)
for itemset, support in list(frequent_itemsets.items())[:5]:
    actual_count = sum(1 for t in transactions if itemset.issubset(frozenset(t)))
    actual_support = actual_count / n_txn
    assert (
        abs(actual_support - support) < 0.005
    ), f"Computed support {support:.4f} disagrees with actual {actual_support:.4f}"
    assert (
        support >= MIN_SUPPORT - 0.001
    ), f"Itemset with support {support:.4f} should meet min_support={MIN_SUPPORT}"
print("\n[ok] Checkpoint passed — Apriori computes correct support values\n")



## TASK 4 — VISUALISE the L1 -> Lk ladder and top itemsets


In [ ]:
sorted_itemsets = sorted(frequent_itemsets.items(), key=lambda kv: -kv[1])

print("Top 15 frequent itemsets by support:")
print(f"  {'Itemset':<45} {'Support':>8}")
print("  " + "-" * 55)
for itemset, support in sorted_itemsets[:15]:
    print(f"  {format_itemset(itemset):<45} {support:>8.4f}")

# Export to polars for any downstream visualisation
top_df = pl.DataFrame(
    {
        "itemset": [format_itemset(s) for s, _ in sorted_itemsets[:30]],
        "size": [len(s) for s, _ in sorted_itemsets[:30]],
        "support": [float(v) for _, v in sorted_itemsets[:30]],
    }
)
top_df.write_csv(OUTPUT_DIR / "apriori_top_itemsets.csv")
print(f"\n  Saved: {OUTPUT_DIR / 'apriori_top_itemsets.csv'}")

# INTERPRETATION: The L1 level is dense (most of the 25 products appear in
# >= 3% of baskets) because Singapore mini-marts stock fast-moving staples.
# The interesting content is at L2 and L3 — that's where co-purchase
# structure (coffee + condensed milk + sugar) appears. If Apriori stops
# early at L2 for your dataset, your min_support is probably too high.



## TASK 5 — APPLY: FairPrice/Sheng Siong shelf layout optimisation


In [ ]:
# SCENARIO: A Singapore supermarket chain operates ~200 neighbourhood
# outlets, each stocking roughly 8,000 SKUs in 1,200 sqft of HDB floor
# space. Shelf real estate is the single largest cost driver after payroll.
#
# A merchandising analyst wants to find frequent 2- and 3-itemsets so that
# physically adjacent shelving can be re-planned: items that appear together
# in >= 3% of baskets are candidates for cross-category co-location (e.g.,
# moving condensed milk next to the kopi aisle, rather than in the dairy
# fridge on the far wall).
#
# WHY APRIORI FITS:
#   - The item universe (~8K SKUs) is too large for brute-force enumeration
#   - Store managers need the frequent-itemset list monthly, not live
#   - The anti-monotone pruning removes ~99.9% of candidate itemsets
#   - Output is directly interpretable — each row is a physical product set
#
# BUSINESS IMPACT: An internal study at a tier-1 Singapore grocer found
# that re-locating the top 50 cross-category frequent pairs into adjacent
# shelving lifted basket size 4-7% without any price change. On annual
# GMV of S$250M, that's a S$10-17M uplift — for zero marginal inventory
# cost. The Apriori run takes seconds; the merchandising re-plan takes a
# weekend.
#
# LIMITATIONS:
#   - Apriori re-scans the transaction log at every level; for 100K+ txns
#     with thousands of SKUs, FP-Growth (Exercise 5.2) is much faster
#   - Support alone is not actionable — you also need confidence + lift
#     (Exercise 5.3) before deciding which pairings are worth the shelf move



## REFLECTION


[x] Implemented the Apriori algorithm from scratch
  [x] Applied the anti-monotone pruning principle in _generate_candidates()
  [x] Counted support with one pass per level against 2,500 baskets
  [x] Identified a production scenario (SG grocery shelf layout) where
      Apriori is the economically optimal choice

  KEY INSIGHT: Pruning > optimisation. The reason Apriori scales is not
  clever data structures — it is the mathematical observation that
  infrequent sets cannot become frequent when you add items. Every
  frequent-itemset miner you'll meet later (FP-Growth, ECLAT) uses the
  same principle, just with different data layouts.

  Next: 02_fp_growth.py — use mlxtend's FP-Growth (no candidate
  generation) and compare it against the Apriori output from this file.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

